In [1]:
import pandas as pd

In [3]:
df = pd.read_excel('Sprint_History.xlsx', sheet_name='Raw Data')

In [4]:
df

,Work Item Type,ID,Title,State,Iteration Path,Resolved Date,Story Points,Activated Date,Created Date,PICategory
0,User Story,485934,Configure the Casting Model in MES,Resolved,DSM-Firmenich\Sprint 01,2025-04-09 10:47:08,2.0,2025-04-08 14:11:07,2025-03-20 09:48:37,Modelling & Configuration
1,User Story,485932,Configure the Final Packaging Model in MES,Resolved,DSM-Firmenich\Sprint 01,2025-04-09 10:47:35,1.0,2025-04-08 14:11:00,2025-03-20 09:47:16,Modelling & Configuration
2,User Story,485927,Master Data Configuration for Pouching in MES,Resolved,DSM-Firmenich\Sprint 01,2025-04-09 10:48:37,1.0,2025-04-08 14:10:53,2025-03-20 09:37:17,Modelling & Configuration
3,User Story,485930,Configure Site Model for HA-CMC Powder & Raw M...,Resolved,DSM-Firmenich\Sprint 01,2025-04-09 12:59:56,1.0,2025-04-09 10:48:12,2025-03-20 09:43:10,Modelling & Configuration
4,User Story,485924,Execute the Weighing Process - IoT W&D,Resolved,DSM-Firmenich\Sprint 01,2025-04-09 16:11:27,8.0,2025-03-27 14:04:13,2025-03-20 09:27:29,Integration - Equipment / Connect IoT
...,...,...,...,...,...,...,...,...,...,...
370,User Story,579449,Sprint 22 - Release Management,Resolved,DSM-Firmenich\Sprint 22,2026-06-02 12:00:22,NaN,2026-06-02 09:17:31,2026-05-21 09:32:17,Release Management
371,User Story,581720,Sprint 23 - Release Management,Resolved,DSM-Firmenich\Sprint 23,2026-06-15 17:27:21,NaN,2026-06-15 17:27:21,2026-06-05 11:02:02,Release Management
372,User Story,562436,"[ENHANCEMENT] Improve EBR - Part I - Generic, ...",Resolved,DSM-Firmenich\Sprint 23,2026-06-17 16:56:19,13.0,2026-06-09 08:33:37,2026-02-10 13:07:44,Customization - Business Logic
373,Bug,581548,Combine with Partial Track-Outs,Resolved,DSM-Firmenich\Sprint 24,2026-06-24 13:40:50,5.0,2026-06-24 09:13:49,2026-06-03 16:48:58,Customization - Business Logic


In [17]:
df['Sprint'] = df['Iteration Path'].apply(lambda x: x.split('\\')[-1])
df['Story Points'] = df['Story Points'].fillna(0).astype(int)
df = df[df['State'].isin(['Resolved', 'Done', 'Closed'])]

In [63]:
# Sprint Summary
import datetime as dt
sprint_df = df.groupby('Sprint')[['Story Points']].sum().reset_index()
sprint_df['Items Resolved'] = df.groupby('Sprint')['ID'].count().values
sprint_df['Last Resolved Date'] = df.groupby('Sprint')['Resolved Date'].max().values
sprint_df['Recency Rank'] = sprint_df['Last Resolved Date'].rank(ascending=False, method='min').astype(int)
# sprint_df['Last Resolved Date'] = sprint_df['Last Resolved Date'].dt.date.astype('datetime64[ns]')
sprint_df

,Sprint,Story Points,Items Resolved,Last Resolved Date,Recency Rank
0,Sprint 01,18,7,2025-08-22 17:38:46,17
1,Sprint 02,53,19,2025-07-07 09:49:15,22
2,Sprint 03,39,30,2025-07-13 20:18:29,21
3,Sprint 04,38,23,2025-05-22 14:55:59,24
4,Sprint 05,23,9,2025-06-04 18:54:06,23
5,Sprint 06,37,19,2025-08-06 14:53:12,19
6,Sprint 07,34,27,2025-07-13 20:19:50,20
7,Sprint 08,47,20,2025-12-02 10:00:59,9
8,Sprint 09,38,28,2025-12-02 10:00:59,9
9,Sprint 10,35,16,2025-08-14 13:21:44,18


In [64]:
# Exclude latest sprints from sampling
exclude_from_sampling = 1
sprint_df = sprint_df[sprint_df['Recency Rank'] > exclude_from_sampling]
sprint_df

,Sprint,Story Points,Items Resolved,Last Resolved Date,Recency Rank
0,Sprint 01,18,7,2025-08-22 17:38:46,17
1,Sprint 02,53,19,2025-07-07 09:49:15,22
2,Sprint 03,39,30,2025-07-13 20:18:29,21
3,Sprint 04,38,23,2025-05-22 14:55:59,24
4,Sprint 05,23,9,2025-06-04 18:54:06,23
5,Sprint 06,37,19,2025-08-06 14:53:12,19
6,Sprint 07,34,27,2025-07-13 20:19:50,20
7,Sprint 08,47,20,2025-12-02 10:00:59,9
8,Sprint 09,38,28,2025-12-02 10:00:59,9
9,Sprint 10,35,16,2025-08-14 13:21:44,18


In [81]:
# Construct a DF of story points for each sprint, for 1000 simulations
import random

simulations = 1000
max_number_sprints = 40

columns: list[str] = [f'S{i+1}' for i in range(max_number_sprints)]
rows: list[list[int]] = []

for i in range(simulations):
    acc = 0
    current_simulation = [(acc := acc + sprint_df['Items Resolved'].sample(n=1, random_state=random.randint(0, 10000)).values[0]) for i in range(max_number_sprints)]
    rows.append(current_simulation)

simulation_df = pd.DataFrame(rows, columns=columns)
simulation_df

,S1,S2,S3,S4,S5,S6,S7,S8,S9,S10,...,S31,S32,S33,S34,S35,S36,S37,S38,S39,S40
0,9,28,30,37,53,85,100,119,138,159,...,518,522,542,544,567,591,606,608,635,644
1,4,9,37,54,58,74,90,110,131,133,...,477,509,537,558,577,598,614,617,619,638
2,30,58,60,79,86,102,126,145,175,180,...,494,514,518,537,548,557,560,592,594,596
3,23,26,45,65,69,97,102,119,146,148,...,490,505,528,552,575,584,587,598,600,617
4,21,40,59,82,97,125,141,157,160,171,...,530,545,550,566,593,609,629,636,656,675
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,5,7,35,67,83,86,97,106,115,132,...,479,499,518,538,559,578,598,601,629,644
996,19,34,39,59,62,66,83,102,105,126,...,466,475,494,513,528,547,556,573,590,614
997,9,28,48,59,68,91,119,122,126,146,...,508,510,514,546,561,568,575,595,606,621
998,2,13,20,37,52,72,104,115,136,164,...,430,441,471,503,505,526,549,570,589,592


In [ ]:
target = 5

targets_df = simulation_df >= target
first_hit = targets_df.idxmax(axis=1)
first_hit[~targets_df.any(axis=1)] = None
first_hit.value_counts().sort_index()

0      S1
1      S2
2      S1
3      S1
4      S1
       ..
995    S1
996    S1
997    S1
998    S2
999    S1
Length: 1000, dtype: str